In [ ]:
import scvi
import scanpy as sc
import numpy as np
from mudata import MuData

In [2]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.3.0


In [ ]:
# read multiomic data
adata = sc.read_h5ad("./Data/RNA_ATAC_ADT/TEA-seq/TEA-seq.h5ad")
adata_gex = adata[:, adata.var['modality'] == "Gene Expression"]
adata_atac = adata[:, adata.var['modality'] == "Peaks"]

In [ ]:
sc.pp.filter_genes(adata_atac, min_cells=int(adata_atac.shape[0] * 0.01))


adata_adt = sc.AnnData(X=adata.obsm['protein_expression'])
adata_adt.obs = adata.obs

In [ ]:
mdata = MuData({"rna": adata_gex, "atac": adata_atac, "protein": adata_adt})
scvi.model.MULTIVI.setup_mudata(mdata, batch_key="batch",
modalities={"rna_layer": "rna", "atac_layer": 'atac', "protein_layer": "protein", "batch_key": "rna"})
model = scvi.model.MULTIVI(mdata)

In [ ]:
model.train()
emb = model.get_latent_representation()
adata_gex.obsm["latent"] = emb

In [ ]:
adata_gex.write("MultiVI_tea_3.h5ad")